# **VERSION 8.0**

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!unzip "/content/drive/MyDrive/BGL Log/BGL.log.zip" -d /content/

Mounted at /content/drive
Archive:  /content/drive/MyDrive/BGL Log/BGL.log.zip
  inflating: /content/BGL.log        


In [2]:
# ==========================================
# CELL 1: REPRODUCIBILITY, IMPORTS & PARSING
# ==========================================

import os
import random
import re
import time
import numpy as np
import pandas as pd
import torch

# ------------------------------
# Reproducibility
# ------------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------
# Persistent paths on Drive (survive runtime restarts)
# ------------------------------
CHECKPOINT_DIR = "/content/drive/MyDrive/BGL Log/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

STANDARD_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_ae_model.pth")
OOV_CKPT_PATH = os.path.join(CHECKPOINT_DIR, "best_ae_oov_aware_model.pth")

PARSED_DF_CACHE_PATH = "/content/drive/MyDrive/BGL Log/parsed_bgl.parquet"

# ------------------------------
# BGL parser
# ------------------------------
bgl_pattern = re.compile(
    r'^(?P<Label>\S+)\s+(?P<Timestamp>\d+)\s+(?P<Date>\d{4}\.\d{2}\.\d{2})\s+'
    r'(?P<Node>\S+)\s+(?P<Time>\d{4}-\d{2}-\d{2}-\d{2}\.\d{2}\.\d{2}\.\d+)\s+'
    r'(?P<NodeRepeat>\S+)\s+(?P<Type>\S+)\s+(?P<Component>\S+)\s+'
    r'(?P<Level>\S+)\s+(?P<Content>.*)$'
)

def extract_bgl_template(content):
    t = re.sub(r'0x[a-fA-F0-9]+', '<*>', content)
    t = re.sub(r'\d+\.\d+\.\d+\.\d+', '<*>', t)
    t = re.sub(
        r'[A-Z0-9]+-[A-Z0-9]+-[A-Z0-9]+-[A-Z0-9]+:[A-Z0-9]+-[A-Z0-9]+',
        '<*>',
        t
    )
    t = re.sub(r'\b\d+\b', '<*>', t)
    return t

if os.path.exists(PARSED_DF_CACHE_PATH):
    print(f"\nFound cached parsed dataframe at {PARSED_DF_CACHE_PATH} — loading instead of re-parsing.")
    df = pd.read_parquet(PARSED_DF_CACHE_PATH)
    print(f"Loaded {len(df):,} parsed logs from cache.")

else:
    file_path = 'BGL.log'
    parsed_logs = []
    start_time = time.time()

    print("\nBeginning parsing of BGL dataset...")

    with open(file_path, 'r') as file:
        for i, line in enumerate(file, start=1):
            match = bgl_pattern.match(line.strip())

            if match:
                parsed_logs.append(match.groupdict())

            if i % 500_000 == 0:
                print(f"Parsed {i:,} lines...")

    df = pd.DataFrame(parsed_logs)

    df['Is_Anomaly'] = df['Label'].apply(
        lambda x: 0 if x == '-' else 1
    )

    df['Event template'] = df['Content'].apply(extract_bgl_template)

    print(f"\nParsing complete: {len(df):,} logs in {time.time() - start_time:.2f} seconds.")

    df.to_parquet(PARSED_DF_CACHE_PATH, index=False)
    print(f"Saved parsed dataframe to {PARSED_DF_CACHE_PATH}")

print("\n--- LOG-LEVEL CLASS DISTRIBUTION ---")
print(df['Is_Anomaly'].value_counts())
print(df['Is_Anomaly'].value_counts(normalize=True))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Found cached parsed dataframe at /content/drive/MyDrive/BGL Log/parsed_bgl.parquet — loading instead of re-parsing.
Loaded 4,713,493 parsed logs from cache.

--- LOG-LEVEL CLASS DISTRIBUTION ---
Is_Anomaly
0    4365033
1     348460
Name: count, dtype: int64
Is_Anomaly
0    0.926072
1    0.073928
Name: proportion, dtype: float64


In [3]:
# ==========================================
# CELL 2: CHRONOLOGICAL SPLIT, VOCABULARY
#          & WINDOW GENERATION
# ==========================================

n_total = len(df)

train_idx = int(n_total * 0.70)
val_idx   = int(n_total * 0.80)

train_df = df.iloc[:train_idx].copy()
val_df   = df.iloc[train_idx:val_idx].copy()
test_df  = df.iloc[val_idx:].copy()

print(
    f"\nChronological split:"
    f"\nTrain: {len(train_df):,}"
    f"\nValidation: {len(val_df):,}"
    f"\nTest: {len(test_df):,}"
)

# ------------------------------
# Train-only vocabulary
# 0 = <UNK>
# ------------------------------
unique_templates = train_df['Event template'].unique()

template_to_id = {
    template: idx + 1
    for idx, template in enumerate(unique_templates)
}

vocab_size = len(template_to_id) + 1

print(f"\nTraining vocabulary size including <UNK>: {vocab_size:,}")

def map_tokens(data):
    return (
        data['Event template']
        .map(template_to_id)
        .fillna(0)
        .astype(int)
        .values
    )

train_df['Event_Token'] = map_tokens(train_df)
val_df['Event_Token']   = map_tokens(val_df)
test_df['Event_Token']  = map_tokens(test_df)

# ------------------------------
# Window creation
# ------------------------------
def create_windows(
    data_subset,
    window_size=20,
    step_size=5
):
    tokens = data_subset['Event_Token'].values
    anomaly_flags = data_subset['Is_Anomaly'].values
    levels = data_subset['Level'].values

    sequences = []
    labels = []
    severity_scores = []

    critical_levels = [
        'WARNING',
        'ERROR',
        'SEVERE',
        'FATAL',
        'FAILURE'
    ]

    for i in range(
        0,
        len(tokens) - window_size + 1,
        step_size
    ):
        seq = tokens[i:i + window_size]

        if len(seq) < window_size:
            continue

        window_is_anomaly = int(
            anomaly_flags[i:i + window_size].sum() > 0
        )

        severity_score = sum(
            level in critical_levels
            for level in levels[i:i + window_size]
        )

        sequences.append(seq)
        labels.append(window_is_anomaly)
        severity_scores.append(severity_score)

    return (
        np.asarray(sequences),
        np.asarray(labels),
        np.asarray(severity_scores)
    )

# ------------------------------
# Overlapping windows
# ------------------------------
X_train, y_train, train_severity = create_windows(
    train_df,
    window_size=20,
    step_size=5
)

X_val, y_val, val_severity = create_windows(
    val_df,
    window_size=20,
    step_size=5
)

X_test, y_test, test_severity = create_windows(
    test_df,
    window_size=20,
    step_size=5
)

# ------------------------------
# Normal-only data for training
# ------------------------------
X_train_normal = X_train[y_train == 0]
X_val_normal   = X_val[y_val == 0]

print("\n--- WINDOW SHAPES ---")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\n--- NORMAL WINDOWS ---")
print("Train normal:", X_train_normal.shape)
print("Val normal  :", X_val_normal.shape)


Chronological split:
Train: 3,299,445
Validation: 471,349
Test: 942,699

Training vocabulary size including <UNK>: 30,050

--- WINDOW SHAPES ---
X_train: (659886, 20)
X_val  : (94266, 20)
X_test : (188536, 20)

--- NORMAL WINDOWS ---
Train normal: (601507, 20)
Val normal  : (83625, 20)


In [4]:
# ==========================================
# CELL 3: NON-OVERLAPPING TEST SET
#          & OOV DIAGNOSTICS
# ==========================================

# Non-overlapping test windows
X_test_nonoverlap, y_test_nonoverlap, test_severity_nonoverlap = create_windows(
    test_df,
    window_size=20,
    step_size=20
)

print("\n--- NON-OVERLAPPING TEST SET ---")
print("Shape:", X_test_nonoverlap.shape)

print("\nClass distribution:")
print(pd.Series(y_test_nonoverlap).value_counts())

print("\nClass proportions:")
print(
    pd.Series(y_test_nonoverlap)
    .value_counts(normalize=True)
)

# ------------------------------
# OOV diagnostics
# ------------------------------
def print_unk_statistics(name, data):
    tokens = data['Event_Token'].values

    overall_unk = np.mean(tokens == 0)

    normal_mask = data['Is_Anomaly'].values == 0
    anomaly_mask = data['Is_Anomaly'].values == 1

    normal_unk = (
        np.mean(tokens[normal_mask] == 0)
        if normal_mask.sum()
        else 0
    )

    anomaly_unk = (
        np.mean(tokens[anomaly_mask] == 0)
        if anomaly_mask.sum()
        else 0
    )

    print(
        f"{name:<12} | "
        f"Overall: {overall_unk:.4%} | "
        f"Normal: {normal_unk:.4%} | "
        f"Anomaly: {anomaly_unk:.4%}"
    )

print("\n--- TOKEN-LEVEL <UNK> STATISTICS ---")

print_unk_statistics("TRAIN", train_df)
print_unk_statistics("VALIDATION", val_df)
print_unk_statistics("TEST", test_df)

print("\n--- WINDOW-LEVEL <UNK> STATISTICS ---")

for name, X_data, y_data in [
    ("Train", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Test", X_test, y_test)
]:

    unk_per_window = (X_data == 0).sum(axis=1)

    normal_windows = y_data == 0
    anomaly_windows = y_data == 1

    print(f"\n{name}")
    print(
        f"All windows with <UNK>: "
        f"{(unk_per_window > 0).mean():.4%}"
    )

    if normal_windows.any():
        print(
            f"Normal windows with <UNK>: "
            f"{(unk_per_window[normal_windows] > 0).mean():.4%}"
        )

    if anomaly_windows.any():
        print(
            f"Anomaly windows with <UNK>: "
            f"{(unk_per_window[anomaly_windows] > 0).mean():.4%}"
        )

print("\n--- NORMAL VALIDATION SEVERITY ---")

normal_val_severity = val_severity[y_val == 0]

print(
    pd.Series(normal_val_severity)
    .value_counts()
    .sort_index()
    .head(10)
)

for p in [90, 95, 97.5, 99]:
    print(
        f"{p}th percentile = "
        f"{np.percentile(normal_val_severity, p)}"
    )


--- NON-OVERLAPPING TEST SET ---
Shape: (47134, 20)

Class distribution:
0    44129
1     3005
Name: count, dtype: int64

Class proportions:
0    0.936246
1    0.063754
Name: proportion, dtype: float64

--- TOKEN-LEVEL <UNK> STATISTICS ---
TRAIN        | Overall: 0.0000% | Normal: 0.0000% | Anomaly: 0.0000%
VALIDATION   | Overall: 45.1472% | Normal: 43.3896% | Anomaly: 61.2592%
TEST         | Overall: 80.0207% | Normal: 79.9085% | Anomaly: 82.1922%

--- WINDOW-LEVEL <UNK> STATISTICS ---

Train
All windows with <UNK>: 0.0000%
Normal windows with <UNK>: 0.0000%
Anomaly windows with <UNK>: 0.0000%

Validation
All windows with <UNK>: 50.0700%
Normal windows with <UNK>: 48.1602%
Anomaly windows with <UNK>: 65.0785%

Test
All windows with <UNK>: 84.9970%
Normal windows with <UNK>: 84.8007%
Anomaly windows with <UNK>: 87.8783%

--- NORMAL VALIDATION SEVERITY ---
0    49014
1      794
2      218
3      179
4      131
5      113
6       95
7      101
8       87
9       96
Name: count, dtype: i

In [5]:
# ==========================================
# CELL 4: PYTORCH SETUP & LSTM AUTOENCODER
# ==========================================

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    DataLoader,
    TensorDataset
)

batch_size = 512

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------
# DataLoaders
# ------------------------------
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(
            X_train_normal,
            dtype=torch.long
        ),
        torch.tensor(
            X_train_normal,
            dtype=torch.long
        )
    ),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(
        torch.tensor(
            X_val_normal,
            dtype=torch.long
        ),
        torch.tensor(
            X_val_normal,
            dtype=torch.long
        )
    ),
    batch_size=batch_size,
    shuffle=False
)

# ------------------------------
# LSTM sequence autoencoder
# ------------------------------
class LogAutoencoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=16,
        hidden_dim=32
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.encoder = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.decoder = nn.LSTM(
            hidden_dim,
            embedding_dim,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, x):

        embedded = self.embedding(x)

        _, (hidden, _) = self.encoder(
            embedded
        )

        hidden_repeated = (
            hidden
            .repeat(x.size(1), 1, 1)
            .transpose(0, 1)
        )

        decoded, _ = self.decoder(
            hidden_repeated
        )

        return self.output_layer(decoded)

print("\nModel architecture:")
print(LogAutoencoder(vocab_size))

Device: cuda
GPU: Tesla T4

Model architecture:
LogAutoencoder(
  (embedding): Embedding(30050, 16)
  (encoder): LSTM(16, 32, batch_first=True)
  (decoder): LSTM(32, 16, batch_first=True)
  (output_layer): Linear(in_features=16, out_features=30050, bias=True)
)


In [6]:
# ==========================================
# CELL 5: TRAIN STANDARD LSTM AUTOENCODER
# ==========================================

criterion = nn.CrossEntropyLoss()

eval_criterion = nn.CrossEntropyLoss(
    reduction='none'
)

model = LogAutoencoder(
    vocab_size=vocab_size,
    embedding_dim=16,
    hidden_dim=32
).to(device)

if os.path.exists(STANDARD_CKPT_PATH):
    print(f"\nFound existing checkpoint at {STANDARD_CKPT_PATH} — loading instead of retraining.")

    model.load_state_dict(
        torch.load(
            STANDARD_CKPT_PATH,
            map_location=device
        )
    )

    model.eval()

    print("Best Standard LSTM AE checkpoint loaded.")

else:
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    max_epochs = 30
    patience = 3
    min_delta = 1e-4

    best_val_loss = float('inf')
    patience_counter = 0

    train_losses = []
    val_losses = []

    print(
        f"\nStarting Standard LSTM AE training on {device}..."
    )

    for epoch in range(max_epochs):

        # ------------------------------
        # Training
        # ------------------------------
        model.train()

        total_train_loss = 0.0

        for data, target in train_loader:

            data = data.to(device)
            target = target.to(device)

            optimizer.zero_grad()

            output = model(data)

            loss = criterion(
                output.view(-1, vocab_size),
                target.view(-1)
            )

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = (
            total_train_loss / len(train_loader)
        )

        train_losses.append(avg_train_loss)

        # ------------------------------
        # Validation
        # ------------------------------
        model.eval()

        total_val_loss = 0.0

        with torch.no_grad():

            for data, target in val_loader:

                data = data.to(device)
                target = target.to(device)

                output = model(data)

                val_loss = criterion(
                    output.view(-1, vocab_size),
                    target.view(-1)
                )

                total_val_loss += val_loss.item()

        avg_val_loss = (
            total_val_loss / len(val_loader)
        )

        val_losses.append(avg_val_loss)

        print(
            f"Epoch [{epoch + 1:02d}/{max_epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

        # ------------------------------
        # Early stopping
        # ------------------------------
        if avg_val_loss < best_val_loss - min_delta:

            best_val_loss = avg_val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                STANDARD_CKPT_PATH
            )

        else:

            patience_counter += 1

            if patience_counter >= patience:

                print(
                    f"Early stopping triggered at "
                    f"Epoch {epoch + 1}"
                )
                break

    # ------------------------------
    # Load best checkpoint
    # ------------------------------
    model.load_state_dict(
        torch.load(
            STANDARD_CKPT_PATH,
            map_location=device
        )
    )

    model.eval()

    print("\nBest Standard LSTM AE checkpoint loaded.")


Found existing checkpoint at /content/drive/MyDrive/BGL Log/checkpoints/best_ae_model.pth — loading instead of retraining.
Best Standard LSTM AE checkpoint loaded.


In [7]:
# ==========================================
# CELL 6: TRAIN OOV-ROBUST DENOISING LSTM AE
# ==========================================

# This model is trained on normal sequences only.
# Input is corrupted with random <UNK>=0 tokens.
# Target remains the original clean sequence.

OOV_MASK_PROB = 0.15

oov_model = LogAutoencoder(
    vocab_size=vocab_size,
    embedding_dim=16,
    hidden_dim=32
).to(device)

def corrupt_with_oov(x, mask_prob=0.15):

    x_corrupt = x.clone()

    # Never corrupt the already-OOV token.
    mask = (
        (torch.rand_like(
            x_corrupt,
            dtype=torch.float32
        ) < mask_prob)
        &
        (x_corrupt != 0)
    )

    x_corrupt[mask] = 0

    return x_corrupt

if os.path.exists(OOV_CKPT_PATH):
    print(f"\nFound existing checkpoint at {OOV_CKPT_PATH} — loading instead of retraining.")

    oov_model.load_state_dict(
        torch.load(
            OOV_CKPT_PATH,
            map_location=device
        )
    )

    oov_model.eval()

    print("Best OOV-Robust checkpoint loaded.")

else:
    oov_optimizer = optim.Adam(
        oov_model.parameters(),
        lr=0.001
    )

    oov_criterion = nn.CrossEntropyLoss()

    max_epochs = 30
    patience = 3
    min_delta = 1e-4

    best_oov_val_loss = float('inf')
    oov_patience_counter = 0

    oov_train_losses = []
    oov_val_losses = []

    print(
        f"\nStarting OOV-Robust LSTM AE training "
        f"with mask probability = {OOV_MASK_PROB:.2f}"
    )

    for epoch in range(max_epochs):

        # ------------------------------
        # Training
        # ------------------------------
        oov_model.train()

        total_train_loss = 0.0

        for clean_data, target in train_loader:

            clean_data = clean_data.to(device)
            target = target.to(device)

            corrupted_data = corrupt_with_oov(
                clean_data,
                OOV_MASK_PROB
            )

            oov_optimizer.zero_grad()

            output = oov_model(
                corrupted_data
            )

            loss = oov_criterion(
                output.view(-1, vocab_size),
                target.view(-1)
            )

            loss.backward()
            oov_optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = (
            total_train_loss / len(train_loader)
        )

        oov_train_losses.append(
            avg_train_loss
        )

        # ------------------------------
        # Validation
        # ------------------------------
        oov_model.eval()

        total_val_loss = 0.0

        with torch.no_grad():

            for clean_data, target in val_loader:

                clean_data = clean_data.to(device)
                target = target.to(device)

                # Validation remains clean.
                output = oov_model(clean_data)

                val_loss = oov_criterion(
                    output.view(-1, vocab_size),
                    target.view(-1)
                )

                total_val_loss += val_loss.item()

        avg_val_loss = (
            total_val_loss / len(val_loader)
        )

        oov_val_losses.append(
            avg_val_loss
        )

        print(
            f"Epoch [{epoch + 1:02d}/{max_epochs}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f}"
        )

        # ------------------------------
        # Early stopping
        # ------------------------------
        if avg_val_loss < best_oov_val_loss - min_delta:

            best_oov_val_loss = avg_val_loss
            oov_patience_counter = 0

            torch.save(
                oov_model.state_dict(),
                OOV_CKPT_PATH
            )

        else:

            oov_patience_counter += 1

            if oov_patience_counter >= patience:

                print(
                    f"OOV-Robust early stopping triggered "
                    f"at Epoch {epoch + 1}"
                )
                break

    # ------------------------------
    # Load best OOV checkpoint
    # ------------------------------
    oov_model.load_state_dict(
        torch.load(
            OOV_CKPT_PATH,
            map_location=device
        )
    )

    oov_model.eval()

    print(
        "\nBest OOV-Robust checkpoint loaded."
    )


Found existing checkpoint at /content/drive/MyDrive/BGL Log/checkpoints/best_ae_oov_aware_model.pth — loading instead of retraining.
Best OOV-Robust checkpoint loaded.


In [8]:
# ==========================================
# CELL 7: SYNTHETIC SUBSTRATE & DISJOINT
#          REAL TEST SET
# ==========================================

SYNTHETIC_SEED = 123
TARGET_SYNTHETIC_WINDOWS = 7500

# --------------------------------------------------
# 1. Identify OOV-free normal non-overlapping windows
# --------------------------------------------------
oov_free_mask_nonoverlap = (
    (y_test_nonoverlap == 0)
    &
    (X_test_nonoverlap != 0).all(axis=1)
)

valid_indices = np.flatnonzero(
    oov_free_mask_nonoverlap
)

available_count = len(valid_indices)

print(
    f"OOV-free normal non-overlapping windows available: "
    f"{available_count:,}"
)

if available_count < 2:
    raise ValueError(
        "Not enough OOV-free normal non-overlapping "
        "windows to build the synthetic substrate."
    )

# --------------------------------------------------
# 2. Use up to 7,500 windows.
#    Force an even number for a 50/50 split.
# --------------------------------------------------
n_clean = min(
    TARGET_SYNTHETIC_WINDOWS,
    available_count
)

n_clean -= n_clean % 2

print(
    f"Using {n_clean:,} OOV-free windows "
    f"for the synthetic substrate."
)

if n_clean < TARGET_SYNTHETIC_WINDOWS:
    print(
        f"Note: only {available_count:,} were available, "
        f"so the experiment uses {n_clean:,} windows."
    )

# --------------------------------------------------
# 3. Randomly select source indices
# --------------------------------------------------
rng_pool = np.random.default_rng(
    SYNTHETIC_SEED
)

selected_positions = rng_pool.choice(
    available_count,
    size=n_clean,
    replace=False
)

synthetic_substrate_indices = (
    valid_indices[selected_positions]
)

# Preserve the selected source-window order.
clean_windows = (
    X_test_nonoverlap[
        synthetic_substrate_indices
    ].copy()
)

# --------------------------------------------------
# 4. Remove those exact source indices from
#    the secondary real-test evaluation
# --------------------------------------------------
real_test_mask = np.ones(
    len(X_test_nonoverlap),
    dtype=bool
)

real_test_mask[
    synthetic_substrate_indices
] = False

X_test_nonoverlap_real = (
    X_test_nonoverlap[
        real_test_mask
    ]
)

y_test_nonoverlap_real = (
    y_test_nonoverlap[
        real_test_mask
    ]
)

# --------------------------------------------------
# 5. SOURCE-INDEX DISJOINTNESS CHECK
# --------------------------------------------------
remaining_indices = np.flatnonzero(
    real_test_mask
)

selected_index_set = set(
    synthetic_substrate_indices.tolist()
)

remaining_index_set = set(
    remaining_indices.tolist()
)

index_overlap = (
    selected_index_set
    .intersection(remaining_index_set)
)

assert len(index_overlap) == 0, (
    "Synthetic substrate and real-test set "
    "share source indices."
)

# Verify clean_windows exactly corresponds to
# the selected source windows.
assert np.array_equal(
    clean_windows,
    X_test_nonoverlap[
        synthetic_substrate_indices
    ]
), (
    "clean_windows does not match the selected "
    "source windows."
)

print(
    "\nSource-index disjointness check: PASSED"
)

# --------------------------------------------------
# 6. Print final dataset information
# --------------------------------------------------
print(
    f"Synthetic substrate size: "
    f"{len(clean_windows):,}"
)

print(
    f"Remaining disjoint real-test windows: "
    f"{len(X_test_nonoverlap_real):,}"
)

print(
    "\nDisjoint real-test class distribution:"
)

print(
    pd.Series(
        y_test_nonoverlap_real
    ).value_counts()
)

print(
    "\nDisjoint real-test class proportions:"
)

print(
    pd.Series(
        y_test_nonoverlap_real
    ).value_counts(normalize=True)
)

# --------------------------------------------------
# 7. Informational duplicate-content check
# --------------------------------------------------
unique_synthetic = np.unique(
    clean_windows,
    axis=0
).shape[0]

unique_real = np.unique(
    X_test_nonoverlap_real,
    axis=0
).shape[0]

print(
    "\nUnique sequence contents:"
)

print(
    f"Synthetic: {unique_synthetic:,} / "
    f"{len(clean_windows):,}"
)

print(
    f"Real test: {unique_real:,} / "
    f"{len(X_test_nonoverlap_real):,}"
)

print(
    "\nNote: identical sequence contents across "
    "different source windows are allowed."
)

OOV-free normal non-overlapping windows available: 6,694
Using 6,694 OOV-free windows for the synthetic substrate.
Note: only 6,694 were available, so the experiment uses 6,694 windows.

Source-index disjointness check: PASSED
Synthetic substrate size: 6,694
Remaining disjoint real-test windows: 40,440

Disjoint real-test class distribution:
0    37435
1     3005
Name: count, dtype: int64

Disjoint real-test class proportions:
0    0.925692
1    0.074308
Name: proportion, dtype: float64

Unique sequence contents:
Synthetic: 1,031 / 6,694
Real test: 2,910 / 40,440

Note: identical sequence contents across different source windows are allowed.


In [9]:
# ==========================================
# CELL 8: SYNTHETIC ANOMALY INJECTION
#          ROBUST SHUFFLE VERSION
# ==========================================

# Known training-vocabulary events only
vocab, counts_ = np.unique(
    X_train_normal[
        X_train_normal != 0
    ],
    return_counts=True
)

# Rarest 10% of known templates
rare_pool = vocab[
    np.argsort(counts_)[
        :max(1, len(vocab) // 10)
    ]
]


def _mask(
    n,
    W,
    frac,
    rng,
    min_k=1
):
    """
    Select exactly k positions per window.
    """

    k = max(
        min_k,
        int(round(frac * W))
    )

    if k > W:
        raise ValueError(
            f"k={k} exceeds window size W={W}"
        )

    ranks = (
        rng.random((n, W))
        .argsort(axis=1)
        .argsort(axis=1)
    )

    return ranks < k


def _shuffle_window(
    window,
    k,
    rng
):
    """
    Shuffle exactly k positions while guaranteeing
    that the resulting window changes whenever the
    original window contains at least two distinct
    event IDs.

    Returns:
        changed_window or None if impossible.
    """

    W = len(window)

    unique_positions = np.flatnonzero(
        np.bincount(
            window
        ) > 0
    )

    # A window containing only one unique event
    # cannot be changed by pure permutation.
    if len(unique_positions) <= 1:
        return None

    # Find two positions containing different values.
    first_pos = None
    second_pos = None

    for i in range(W):
        for j in range(i + 1, W):
            if window[i] != window[j]:
                first_pos = i
                second_pos = j
                break

        if first_pos is not None:
            break

    if first_pos is None:
        return None

    # Start with two unequal positions.
    selected = [first_pos, second_pos]

    # Fill remaining positions randomly.
    if k > 2:

        remaining = np.array(
            [
                p for p in range(W)
                if p not in selected
            ],
            dtype=int
        )

        extra = rng.choice(
            remaining,
            size=k - 2,
            replace=False
        )

        selected.extend(
            extra.tolist()
        )

    selected = np.asarray(
        selected,
        dtype=int
    )

    # Rotate selected values.
    # Because selected[0] and selected[1]
    # contain different values, this guarantees
    # at least one position changes.
    result = window.copy()

    values = window[selected].copy()

    result[selected] = np.roll(
        values,
        1
    )

    if np.array_equal(
        result,
        window
    ):
        raise RuntimeError(
            "Shuffle construction unexpectedly "
            "produced no change."
        )

    return result


def inject(
    X,
    kind,
    frac,
    rng
):
    """
    Apply a length-preserving perturbation.

    Only windows whose contents actually changed
    are returned.
    """

    X = np.asarray(X)

    Xp = X.copy()

    n, W = Xp.shape

    k = max(
        1,
        int(round(frac * W))
    )

    if k > W:
        raise ValueError(
            f"Perturbation size k={k} "
            f"exceeds window size W={W}"
        )

    # --------------------------------------------------
    # Standard position mask for all perturbations
    # except shuffle.
    # --------------------------------------------------
    if kind == "shuffle":

        # Shuffle is handled separately because we
        # need to guarantee a true permutation change.
        changed_windows = []

        for i in range(n):

            changed = _shuffle_window(
                X[i],
                k=max(2, k),
                rng=rng
            )

            if changed is not None:
                changed_windows.append(
                    changed
                )

        if not changed_windows:
            return X[:0].copy()

        return np.asarray(
            changed_windows
        )

    # --------------------------------------------------
    # Other perturbations
    # --------------------------------------------------
    m = _mask(
        n,
        W,
        frac,
        rng,
        min_k=1
    )

    if kind == "substitute":

        # Replace selected positions with random
        # known training-vocabulary events.
        Xp[m] = rng.choice(
            vocab,
            size=m.sum()
        )

    elif kind == "rare":

        # Replace with rarely occurring known events.
        Xp[m] = rng.choice(
            rare_pool,
            size=m.sum()
        )

    elif kind == "oov":

        # Replace selected events with <UNK>=0.
        Xp[m] = 0

    elif kind == "repeat":

        # Copy ORIGINAL left neighbour.
        # No cascading mutation.
        for j in range(1, W):

            Xp[
                m[:, j],
                j
            ] = X[
                m[:, j],
                j - 1
            ]

    else:

        raise ValueError(
            f"Unknown perturbation type: {kind}"
        )

    changed = (
        Xp != X
    ).any(axis=1)

    return Xp[changed]


def inject_exact(
    X,
    kind,
    frac,
    rng,
    target_n,
    max_tries=20
):
    """
    Generate exactly target_n changed windows.
    """

    if target_n <= 0:
        return np.empty_like(
            X[:0]
        )

    out = []
    have = 0

    for attempt in range(max_tries):

        shuffled = X[
            rng.permutation(
                len(X)
            )
        ]

        batch = inject(
            shuffled,
            kind,
            frac,
            rng
        )

        if len(batch) > 0:

            out.append(batch)

            have += len(batch)

        if have >= target_n:

            result = np.concatenate(
                out
            )

            return result[:target_n]

    raise RuntimeError(
        f"Could not generate {target_n} "
        f"changed windows for "
        f"{kind}/{frac} after "
        f"{max_tries} attempts. "
        f"Check whether the substrate contains "
        f"enough windows with at least two "
        f"distinct event IDs."
    )

In [10]:
# ==========================================
# CELL 9: THRESHOLD-BASED CLASSICAL BASELINES
# ==========================================

# Training vocabulary only.
V = (
    int(
        X_train_normal.max()
    )
    + 1
)

# Verify no non-training IDs slipped into evaluation sets.
for name, arr in [
    ("validation", X_val_normal),
    ("synthetic", clean_windows),
    ("real test", X_test_nonoverlap_real)
]:

    assert (
        arr.max() < V
    ), (
        f"{name}: found template IDs "
        f"outside training vocabulary"
    )

print(
    f"Training vocabulary size V = {V:,}"
)

# ==========================================
# A. EVENT RARITY
# ==========================================

freq = (
    np.bincount(
        X_train_normal.ravel(),
        minlength=V
    )
    + 1.0
)

# OOV is intentionally treated as
# maximally surprising.
freq[0] = 1.0

neglog_freq = -np.log(
    freq / freq.sum()
)

def score_rarity(X):

    return neglog_freq[X].mean(
        axis=1
    )


# ==========================================
# B. BIGRAM TRANSITION
# ==========================================

_k = []
_c = []

for i in range(
    0,
    len(X_train_normal),
    50_000
):

    b_ = X_train_normal[
        i:i + 50_000
    ].astype(np.int64)

    a_ = b_[:, :-1].ravel()
    n_ = b_[:, 1:].ravel()

    ok = (
        (a_ != 0)
        &
        (n_ != 0)
    )

    u, c = np.unique(
        a_[ok] * V + n_[ok],
        return_counts=True
    )

    _k.append(u)
    _c.append(c)

if not _k:

    raise ValueError(
        "No observed bigram chunks found."
    )

_k = np.concatenate(_k)
_c = np.concatenate(_c)

bg_keys, _inv = np.unique(
    _k,
    return_inverse=True
)

if len(bg_keys) == 0:

    raise ValueError(
        "No observed non-OOV bigram "
        "transitions found in training data."
    )

bg_counts = np.bincount(
    _inv,
    weights=_c
)

row_tot = np.bincount(
    bg_keys // V,
    weights=bg_counts,
    minlength=V
)

def score_bigram(
    X,
    chunk=100_000
):

    out = []

    for i in range(
        0,
        len(X),
        chunk
    ):

        Xi = X[
            i:i + chunk
        ].astype(np.int64)

        a = Xi[:, :-1]
        b = Xi[:, 1:]

        key = (
            a * V + b
        )

        pos = np.searchsorted(
            bg_keys,
            key
        )

        pos = np.minimum(
            pos,
            len(bg_keys) - 1
        )

        cnt = np.where(
            bg_keys[pos] == key,
            bg_counts[pos],
            0.0
        )

        prob = (
            cnt + 1.0
        ) / (
            row_tot[a] + V
        )

        # OOV transitions deliberately
        # receive probability 1/V.
        prob = np.where(
            (a == 0) | (b == 0),
            1.0 / V,
            prob
        )

        out.append(
            -np.log(prob).mean(
                axis=1
            )
        )

    return np.concatenate(out)


# ==========================================
# C. COUNT Z-SCORE
# ==========================================

def _counts(X):

    C = np.zeros(
        (len(X), V),
        dtype=np.float32
    )

    rows = np.repeat(
        np.arange(len(X)),
        X.shape[1]
    )

    np.add.at(
        C,
        (
            rows,
            X.ravel()
        ),
        1
    )

    return C


# Around 256 MB per dense float32 block.
CZ_CHUNK = max(
    1,
    min(
        50_000,
        64_000_000 // V
    )
)

s1 = np.zeros(
    V,
    dtype=np.float64
)

s2 = np.zeros(
    V,
    dtype=np.float64
)

N = 0

for i in range(
    0,
    len(X_train_normal),
    CZ_CHUNK
):

    C = _counts(
        X_train_normal[
            i:i + CZ_CHUNK
        ]
    )

    s1 += C.sum(
        0,
        dtype=np.float64
    )

    s2 += (
        C ** 2
    ).sum(
        0,
        dtype=np.float64
    )

    N += len(C)

mu = s1 / N

sd = np.sqrt(
    np.maximum(
        s2 / N - mu ** 2,
        0
    )
) + 1e-3

mu32 = mu.astype(
    np.float32
)

sd32 = sd.astype(
    np.float32
)

def score_countz(X):

    out = []

    for i in range(
        0,
        len(X),
        CZ_CHUNK
    ):

        C = _counts(
            X[
                i:i + CZ_CHUNK
            ]
        )

        out.append(
            np.abs(
                (C - mu32)
                / sd32
            ).max(
                axis=1
            )
        )

    return np.concatenate(
        out
    )

Training vocabulary size V = 30,050


In [11]:
# ==========================================
# CELL 9b: COUNT Z-SCORE NUMERICAL DIAGNOSTICS
# ==========================================

var = np.maximum(s2 / N - mu ** 2, 0.0)

print("\n=== COUNT Z-SCORE DIAGNOSTICS (fit stats) ===")
print("Variance NaN count :", np.isnan(var).sum())
print("Variance Inf count :", np.isinf(var).sum())
print("Zero-variance count:", np.count_nonzero(var == 0))
print("SD NaN count       :", np.isnan(sd).sum())
print("SD Inf count       :", np.isinf(sd).sum())
print("SD min / max       :", sd.min(), sd.max())

print("\n<UNK> (token 0) statistics:")
print("mu[0]              :", mu[0])
print("raw variance[0]    :", var[0])
print("sd[0]              :", sd[0])

print("\nSmall SD counts:")
for eps in [1e-6, 1e-4, 1e-3, 1e-2]:
    print(f"sd <= {eps:g} :", np.count_nonzero(sd <= eps))

# ------------------------------------------------------------
# Test-set score behavior
# ------------------------------------------------------------
cz_test_scores = score_countz(X_test_nonoverlap_real)

print("\n=== COUNT Z TEST SCORE DIAGNOSTICS ===")
print("NaN scores    :", np.isnan(cz_test_scores).sum())
print("Inf scores    :", np.isinf(cz_test_scores).sum())
print("Exactly 20000 :", np.count_nonzero(cz_test_scores == 20000.0))
print("> 10000       :", np.count_nonzero(cz_test_scores > 10000.0))
print("> 5000        :", np.count_nonzero(cz_test_scores > 5000.0))

oov_per_window = (X_test_nonoverlap_real == 0).sum(axis=1)
print("Windows with OOV:", np.count_nonzero(oov_per_window > 0))
print("Max OOV/window  :", oov_per_window.max())

print("\nOOV count, grouped by whether score == 20000:")
print(np.unique(oov_per_window[cz_test_scores == 20000.0], return_counts=True))


=== COUNT Z-SCORE DIAGNOSTICS (fit stats) ===
Variance NaN count : 0
Variance Inf count : 0
Zero-variance count: 789
SD NaN count       : 0
SD Inf count       : 0
SD min / max       : 0.001 9.845809717874216

<UNK> (token 0) statistics:
mu[0]              : 0.0
raw variance[0]    : 0.0
sd[0]              : 0.001

Small SD counts:
sd <= 1e-06 : 0
sd <= 0.0001 : 0
sd <= 0.001 : 789
sd <= 0.01 : 26566

=== COUNT Z TEST SCORE DIAGNOSTICS ===
NaN scores    : 0
Inf scores    : 0
Exactly 20000 : 35888
> 10000       : 37830
> 5000        : 38670
Windows with OOV: 40075
Max OOV/window  : 20

OOV count, grouped by whether score == 20000:
(array([ 0, 20]), array([  313, 35575]))


In [12]:
# ==========================================
# CELL 10: SMOKE-TEST CONFIGURATION
# ==========================================

SMOKE = False

N_SEEDS = (
    1
    if SMOKE
    else 3
)

if SMOKE:

    _r = np.random.default_rng(0)

    # Validation subset
    VAL = X_val_normal[
        :min(2000, len(X_val_normal))
    ]

    # Synthetic substrate subset
    CLEAN = clean_windows[
        :min(2000, len(clean_windows))
    ].copy()

    # Stratified real-test subset
    _pos = np.flatnonzero(
        y_test_nonoverlap_real == 1
    )

    _neg = np.flatnonzero(
        y_test_nonoverlap_real == 0
    )

    assert (
        len(_pos) > 0
        and len(_neg) > 0
    ), (
        "Real test set must contain "
        "both classes."
    )

    n_pos = min(
        len(_pos),
        500
    )

    n_neg = min(
        len(_neg),
        1500
    )

    _idx = np.concatenate(
        [
            _r.choice(
                _pos,
                n_pos,
                replace=False
            ),
            _r.choice(
                _neg,
                n_neg,
                replace=False
            )
        ]
    )

    REAL_X = (
        X_test_nonoverlap_real[
            _idx
        ]
    )

    REAL_Y = (
        y_test_nonoverlap_real[
            _idx
        ]
    )

else:

    VAL = X_val_normal

    CLEAN = clean_windows

    REAL_X = X_test_nonoverlap_real

    REAL_Y = y_test_nonoverlap_real

assert len(VAL) > 0
assert len(CLEAN) > 0
assert len(REAL_X) > 0

print("SMOKE:", SMOKE)
print("Validation windows used:", len(VAL))
print("Synthetic substrate used:", len(CLEAN))
print("Real-test windows used:", len(REAL_X))
print("Seeds:", N_SEEDS)

SMOKE: False
Validation windows used: 83625
Synthetic substrate used: 6694
Real-test windows used: 40440
Seeds: 3


In [13]:
# ==========================================
# CELL 11: SELF-CONTAINED MODEL SCORING
# ==========================================

# Per-token reconstruction loss.
eval_criterion = nn.CrossEntropyLoss(
    reduction="none"
)

def make_score_fn(eval_model):
    """
    Returns a scoring function.

    Per-window anomaly score:
        mean per-token reconstruction loss
    """

    def score_fn(X):

        X = np.asarray(X)

        if len(X) == 0:
            return np.array(
                [],
                dtype=np.float32
            )

        t = torch.as_tensor(
            X,
            dtype=torch.long
        )

        loader = DataLoader(
            TensorDataset(t, t),
            batch_size=batch_size,
            shuffle=False
        )

        losses = []

        eval_model.eval()

        with torch.no_grad():

            for data, target in loader:

                data = data.to(device)
                target = target.to(device)

                output = eval_model(
                    data
                ).transpose(
                    1,
                    2
                )

                token_loss = (
                    eval_criterion(
                        output,
                        target
                    )
                )

                window_loss = (
                    token_loss.mean(
                        dim=1
                    )
                )

                losses.append(
                    window_loss
                    .cpu()
                    .numpy()
                )

        scores = np.concatenate(
            losses
        )

        if len(scores) != len(X):

            raise RuntimeError(
                f"Score length mismatch: "
                f"got {len(scores)}, "
                f"expected {len(X)}"
            )

        if not np.isfinite(scores).all():

            raise RuntimeError(
                "NaN/Inf detected in model scores."
            )

        return scores

    return score_fn


MODELS = {

    "Standard LSTM AE":
        make_score_fn(model),

    "OOV-Robust LSTM AE":
        make_score_fn(oov_model),

    "Event Rarity":
        score_rarity,

    "Bigram Transition":
        score_bigram,

    "Count Z-Score":
        score_countz,
}

In [14]:
# ==========================================
# CELL 12: SMOKE CHECKS & THRESHOLDS
# ==========================================

print("\n--- DETECTOR SMOKE CHECKS ---")

for name, fn in MODELS.items():

    n = min(
        100,
        len(VAL)
    )

    scores = fn(
        VAL[:n]
    )

    assert (
        len(scores) == n
    ), (
        f"{name}: invalid score length"
    )

    assert np.isfinite(
        scores
    ).all(), (
        f"{name}: NaN/Inf scores"
    )

    print(
        f"{name:<24} "
        f"OK | "
        f"range = "
        f"{scores.min():.6f} - "
        f"{scores.max():.6f}"
    )

print("\nSmoke checks passed.")

# ------------------------------
# Confirm the two models differ
# ------------------------------
same_parameters = all(
    torch.equal(
        a,
        b
    )
    for a, b in zip(
        model.parameters(),
        oov_model.parameters()
    )
)

print(
    "\nStandard and OOV-Robust have "
    f"identical parameters: "
    f"{same_parameters}"
)

std_sample = MODELS[
    "Standard LSTM AE"
](
    VAL[:100]
)

oov_sample = MODELS[
    "OOV-Robust LSTM AE"
](
    VAL[:100]
)

print(
    f"Standard score range   : "
    f"{std_sample.min():.6f} - "
    f"{std_sample.max():.6f}"
)

print(
    f"OOV-Robust score range : "
    f"{oov_sample.min():.6f} - "
    f"{oov_sample.max():.6f}"
)

print(
    "Mean |score difference| : "
    f"{np.mean(np.abs(std_sample - oov_sample)):.6f}"
)

# ------------------------------
# 99th-percentile threshold
# per detector
# ------------------------------
THR = {}

print("\n--- 99TH-PERCENTILE THRESHOLDS ---")

for name, fn in MODELS.items():

    val_scores = fn(
        VAL
    )

    threshold = np.percentile(
        val_scores,
        99
    )

    THR[name] = float(
        threshold
    )

    print(
        f"{name:<24} "
        f"{threshold:.6f}"
    )


--- DETECTOR SMOKE CHECKS ---
Standard LSTM AE         OK | range = 2.112517 - 14.395271
OOV-Robust LSTM AE       OK | range = 1.449150 - 16.549696
Event Rarity             OK | range = 5.048918 - 16.305421
Bigram Transition        OK | range = 2.091951 - 10.310695
Count Z-Score            OK | range = 7.883318 - 20000.000000

Smoke checks passed.

Standard and OOV-Robust have identical parameters: False
Standard score range   : 2.112517 - 14.395271
OOV-Robust score range : 1.449150 - 16.549696
Mean |score difference| : 0.571230

--- 99TH-PERCENTILE THRESHOLDS ---
Standard LSTM AE         14.395271
OOV-Robust LSTM AE       15.227727
Event Rarity             16.305421
Bigram Transition        10.310618
Count Z-Score            20000.000000


In [15]:
# ==========================================
# CELL 12b: THRESHOLD-TIE CHECK & PARAMETER COUNT
# ==========================================

# ------------------------------------------------------------
# Count Z-Score threshold-tie check
# (explains zero predicted positives at 99th percentile)
# ------------------------------------------------------------
cz_val = MODELS["Count Z-Score"](VAL)

print("\n=== COUNT Z-SCORE THRESHOLD TIE CHECK ===")
print("99th percentile:", np.percentile(cz_val, 99))
print("Max:", cz_val.max())
print("Number exactly at max:", np.count_nonzero(cz_val == cz_val.max()))
print(
    "Count-Z threshold equals max validation score:",
    np.isclose(THR["Count Z-Score"], cz_val.max())
)

# ------------------------------------------------------------
# Corrected model comparison: parameter count, not exact equality
# (independently trained models should differ; this checks
#  architecture consistency instead)
# ------------------------------------------------------------
std_params = sum(p.numel() for p in model.parameters())
oov_params = sum(p.numel() for p in oov_model.parameters())

print("\n=== MODEL PARAMETER COUNT CHECK ===")
print("Standard parameter count  :", std_params)
print("OOV-Robust parameter count:", oov_params)
print("Same parameter count      :", std_params == oov_params)


=== COUNT Z-SCORE THRESHOLD TIE CHECK ===
99th percentile: 20000.0
Max: 20000.0
Number exactly at max: 33917
Count-Z threshold equals max validation score: True

=== MODEL PARAMETER COUNT CHECK ===
Standard parameter count  : 1001250
OOV-Robust parameter count: 1001250
Same parameter count      : True


In [16]:
# ==========================================
# CELL 13: EVALUATION FUNCTION
# ==========================================

from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    average_precision_score,
    accuracy_score
)

def evaluate(
    name,
    y,
    scores
):

    y = np.asarray(y).astype(bool)
    scores = np.asarray(scores)

    if len(y) != len(scores):

        raise RuntimeError(
            f"{name}: label/score length mismatch "
            f"({len(y)} vs {len(scores)})"
        )

    if not np.isfinite(scores).all():

        raise RuntimeError(
            f"{name}: non-finite scores detected"
        )

    predictions = (
        scores > THR[name]
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y,
            predictions,
            average="binary",
            zero_division=0
        )
    )

    tn = np.sum(
        (~predictions) & (~y)
    )

    fp = np.sum(
        predictions & (~y)
    )

    fn = np.sum(
        (~predictions) & y
    )

    tp = np.sum(
        predictions & y
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn)
        else 0.0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp)
        else 0.0
    )

    accuracy = accuracy_score(
        y,
        predictions
    )

    return {
        "model": name,
        "AUC": roc_auc_score(
            y,
            scores
        ),
        "AP": average_precision_score(
            y,
            scores
        ),
        "Accuracy": accuracy,
        "P": precision,
        "R": recall,
        "F1": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }

In [17]:
# ==========================================
# CELL 14: PRIMARY FULL BGL TEST EVALUATION
# ==========================================

print(
    "\n=== PRIMARY FULL BGL TEST EVALUATION ==="
)

primary_rows = []

for name, fn in MODELS.items():

    print(
        f"Scoring full test set with: {name}"
    )

    scores = fn(
        X_test
    )

    result = evaluate(
        name,
        y_test,
        scores
    )

    primary_rows.append(
        result
    )

primary_results = pd.DataFrame(
    primary_rows
)

print(
    "\n--- PRIMARY RESULTS ---"
)

print(
    primary_results.round(4)
    .to_string(index=False)
)


=== PRIMARY FULL BGL TEST EVALUATION ===
Scoring full test set with: Standard LSTM AE
Scoring full test set with: OOV-Robust LSTM AE
Scoring full test set with: Event Rarity
Scoring full test set with: Bigram Transition
Scoring full test set with: Count Z-Score

--- PRIMARY RESULTS ---
             model    AUC     AP  Accuracy      P      R     F1    FPR    FNR     TN   FP    FN   TP
  Standard LSTM AE 0.5972 0.1836    0.9440 1.0000 0.1221 0.2177 0.0000 0.8779 176508    0 10559 1469
OOV-Robust LSTM AE 0.6532 0.1211    0.9216 0.2692 0.1334 0.1784 0.0247 0.8666 172151 4357 10423 1605
      Event Rarity 0.5684 0.0729    0.9362 0.0000 0.0000 0.0000 0.0000 1.0000 176508    0 12028    0
 Bigram Transition 0.5774 0.0765    0.9363 0.5844 0.0037 0.0074 0.0002 0.9963 176476   32 11983   45
     Count Z-Score 0.5413 0.0686    0.9362 0.0000 0.0000 0.0000 0.0000 1.0000 176508    0 12028    0


In [18]:
# ==========================================
# CELL 15: SECONDARY DISJOINT REAL TEST
# ==========================================

print(
    "\n=== SECONDARY DISJOINT REAL TEST ==="
)

real_rows = []

for name, fn in MODELS.items():

    print(
        f"Scoring disjoint real test with: {name}"
    )

    scores = fn(
        REAL_X
    )

    result = evaluate(
        name,
        REAL_Y,
        scores
    )

    real_rows.append(
        result
    )

real_results = pd.DataFrame(
    real_rows
)

print(
    "\n--- DISJOINT REAL-TEST RESULTS ---"
)

print(
    real_results.round(4)
    .to_string(index=False)
)


=== SECONDARY DISJOINT REAL TEST ===
Scoring disjoint real test with: Standard LSTM AE
Scoring disjoint real test with: OOV-Robust LSTM AE
Scoring disjoint real test with: Event Rarity
Scoring disjoint real test with: Bigram Transition
Scoring disjoint real test with: Count Z-Score

--- DISJOINT REAL-TEST RESULTS ---
             model    AUC     AP  Accuracy      P      R     F1    FPR    FNR    TN   FP   FN  TP
  Standard LSTM AE 0.5263 0.1834    0.9347 1.0000 0.1218 0.2171 0.0000 0.8782 37435    0 2639 366
OOV-Robust LSTM AE 0.5919 0.1216    0.9083 0.2666 0.1338 0.1782 0.0295 0.8662 36329 1106 2603 402
      Event Rarity 0.4924 0.0730    0.9257 0.0000 0.0000 0.0000 0.0000 1.0000 37435    0 3005   0
 Bigram Transition 0.5015 0.0764    0.9258 0.6471 0.0037 0.0073 0.0002 0.9963 37429    6 2994  11
     Count Z-Score 0.4609 0.0688    0.9257 0.0000 0.0000 0.0000 0.0000 1.0000 37435    0 3005   0


In [19]:
# ==========================================
# CELL 16: SYNTHETIC STRESS TESTS
# ==========================================

KINDS = [
    "substitute",
    "rare",
    "oov",
    "repeat",
    "shuffle"
]

FRACS = [
    0.1,
    0.2,
    0.4
]

half = len(CLEAN) // 2

neg = CLEAN[:half]
base = CLEAN[half:]

print(
    "\nSynthetic negatives:",
    len(neg)
)

print(
    "Synthetic positive substrate:",
    len(base)
)

rows = []

for seed_offset in range(
    N_SEEDS
):

    seed_value = (
        SYNTHETIC_SEED
        + seed_offset
    )

    rng = np.random.default_rng(
        seed_value
    )

    print(
        f"\n--- Synthetic seed "
        f"{seed_offset} "
        f"(rng={seed_value}) ---"
    )

    for kind in KINDS:

        for frac in FRACS:

            print(
                f"Generating "
                f"{kind:>10} | "
                f"{frac:.1f}"
            )

            pos = inject_exact(
                base,
                kind,
                frac,
                rng,
                len(neg)
            )

            # Balanced 50/50 synthetic set.
            X_syn = np.concatenate(
                [
                    neg,
                    pos
                ]
            )

            y_syn = np.concatenate(
                [
                    np.zeros(
                        len(neg),
                        dtype=int
                    ),
                    np.ones(
                        len(pos),
                        dtype=int
                    )
                ]
            )

            # Shuffle synthetic order.
            perm = rng.permutation(
                len(X_syn)
            )

            X_syn = X_syn[
                perm
            ]

            y_syn = y_syn[
                perm
            ]

            for name, fn in MODELS.items():

                scores = fn(
                    X_syn
                )

                result = evaluate(
                    name,
                    y_syn,
                    scores
                )

                rows.append(
                    {
                        "seed":
                            seed_offset,
                        "kind":
                            kind,
                        "frac":
                            frac,
                        **result
                    }
                )

stress = pd.DataFrame(
    rows
)

print(
    "\nSynthetic evaluation complete."
)

print(
    "Rows:",
    len(stress)
)


Synthetic negatives: 3347
Synthetic positive substrate: 3347

--- Synthetic seed 0 (rng=123) ---
Generating substitute | 0.1
Generating substitute | 0.2
Generating substitute | 0.4
Generating       rare | 0.1
Generating       rare | 0.2
Generating       rare | 0.4
Generating        oov | 0.1
Generating        oov | 0.2
Generating        oov | 0.4
Generating     repeat | 0.1
Generating     repeat | 0.2
Generating     repeat | 0.4
Generating    shuffle | 0.1
Generating    shuffle | 0.2
Generating    shuffle | 0.4

--- Synthetic seed 1 (rng=124) ---
Generating substitute | 0.1
Generating substitute | 0.2
Generating substitute | 0.4
Generating       rare | 0.1
Generating       rare | 0.2
Generating       rare | 0.4
Generating        oov | 0.1
Generating        oov | 0.2
Generating        oov | 0.4
Generating     repeat | 0.1
Generating     repeat | 0.2
Generating     repeat | 0.4
Generating    shuffle | 0.1
Generating    shuffle | 0.2
Generating    shuffle | 0.4

--- Synthetic seed 2 (rng

In [20]:
# ==========================================
# CELL 17: SYNTHETIC SUMMARY
# ==========================================

summary = (
    stress
    .groupby(
        [
            "kind",
            "frac",
            "model"
        ]
    )[
        [
            "AUC",
            "AP",
            "Accuracy",
            "P",
            "R",
            "F1",
            "FPR",
            "FNR"
        ]
    ]
    .agg(
        [
            "mean",
            "std"
        ]
    )
    .round(4)
)

print(
    "\n=== SYNTHETIC STRESS TEST SUMMARY ==="
)

print(
    summary.to_string()
)


=== SYNTHETIC STRESS TEST SUMMARY ===
                                       AUC              AP         Accuracy               P               R              F1          FPR          FNR        
                                      mean     std    mean     std     mean     std    mean     std    mean     std    mean     std mean  std    mean     std
kind       frac model                                                                                                                                        
oov        0.1  Bigram Transition   0.8980  0.0002  0.8079  0.0005   0.5003  0.0000  1.0000  0.0000  0.0006  0.0000  0.0012  0.0000  0.0  0.0  0.9994  0.0000
                Count Z-Score       1.0000  0.0000  1.0000  0.0000   0.5000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0  0.0  1.0000  0.0000
                Event Rarity        0.8731  0.0000  0.7844  0.0003   0.5000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0000  0.0  0.0  1.0000  0.0000
             

In [21]:
# ==========================================
# CELL 18: SAVE RESULTS
# ==========================================

suffix = (
    "_smoke"
    if SMOKE
    else ""
)

primary_results.to_csv(
    f"primary_full_bgl_results{suffix}.csv",
    index=False
)

real_results.to_csv(
    f"real_test_results{suffix}.csv",
    index=False
)

stress.to_csv(
    f"stress_test_results{suffix}.csv",
    index=False
)

# Save thresholds too
threshold_df = pd.DataFrame(
    [
        {
            "model": name,
            "threshold_percentile": 99,
            "threshold": threshold
        }
        for name, threshold
        in THR.items()
    ]
)

threshold_df.to_csv(
    f"detector_thresholds{suffix}.csv",
    index=False
)

print(
    "\nFiles saved:"
)

print(
    f"primary_full_bgl_results{suffix}.csv"
)

print(
    f"real_test_results{suffix}.csv"
)

print(
    f"stress_test_results{suffix}.csv"
)

print(
    f"detector_thresholds{suffix}.csv"
)


Files saved:
primary_full_bgl_results.csv
real_test_results.csv
stress_test_results.csv
detector_thresholds.csv


In [22]:
# ==========================================
# CELL 19: COMPACT REPORT FOR PAPER / PPT
# ==========================================

print("\n" + "=" * 90)
print("FINAL EXPERIMENT REPORT")
print("=" * 90)

print("\n1. PRIMARY FULL BGL TEST")
print(
    primary_results[
        [
            "model",
            "AUC",
            "AP",
            "Accuracy",
            "P",
            "R",
            "F1",
            "FPR",
            "FNR"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n2. SECONDARY DISJOINT REAL TEST")
print(
    real_results[
        [
            "model",
            "AUC",
            "AP",
            "Accuracy",
            "P",
            "R",
            "F1",
            "FPR",
            "FNR"
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print("\n3. THRESHOLDS")
print(
    threshold_df.round(6)
    .to_string(index=False)
)

print("\n4. MODEL PARAMETER CHECK")
print(
    "Standard and OOV-Robust identical:",
    same_parameters
)

print("\n" + "=" * 90)


FINAL EXPERIMENT REPORT

1. PRIMARY FULL BGL TEST
             model    AUC     AP  Accuracy      P      R     F1    FPR    FNR
  Standard LSTM AE 0.5972 0.1836    0.9440 1.0000 0.1221 0.2177 0.0000 0.8779
OOV-Robust LSTM AE 0.6532 0.1211    0.9216 0.2692 0.1334 0.1784 0.0247 0.8666
      Event Rarity 0.5684 0.0729    0.9362 0.0000 0.0000 0.0000 0.0000 1.0000
 Bigram Transition 0.5774 0.0765    0.9363 0.5844 0.0037 0.0074 0.0002 0.9963
     Count Z-Score 0.5413 0.0686    0.9362 0.0000 0.0000 0.0000 0.0000 1.0000

2. SECONDARY DISJOINT REAL TEST
             model    AUC     AP  Accuracy      P      R     F1    FPR    FNR
  Standard LSTM AE 0.5263 0.1834    0.9347 1.0000 0.1218 0.2171 0.0000 0.8782
OOV-Robust LSTM AE 0.5919 0.1216    0.9083 0.2666 0.1338 0.1782 0.0295 0.8662
      Event Rarity 0.4924 0.0730    0.9257 0.0000 0.0000 0.0000 0.0000 1.0000
 Bigram Transition 0.5015 0.0764    0.9258 0.6471 0.0037 0.0073 0.0002 0.9963
     Count Z-Score 0.4609 0.0688    0.9257 0.0000 0.0000 0